# Graphe Hydrographique

## Import des librairies

In [2]:
import pathlib
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pyproj
import networkx as nx
from shapely import Point, LineString, MultiLineString, GeometryCollection
from shapely.ops import split, snap, linemerge
from tqdm.auto import tqdm

## Tentative avec Cours d'eau comme base

### Pré traitement des données

In [5]:
gdf = gpd.read_file("données hackaton/CoursEau/CoursEau_FXX.shp")

Code_OH = "03C00000020008"

m = gdf['CdOH'].str.startswith(Code_OH)
seine = gdf[m].copy()
seine.head(3)


,gid,CdOH,TopoOH,SourceNomO,DateCreati,StatutOH,InfluenceM,CaractereP,Commentair,ProjCoordO,geometry
5,120733,03C0000002000849015,Vallée des Lavoirs,IGN;BDCarthage,2017/01/24 16:06:18.918,Validé,NaN,NaN,None,RGF93 / Lambert 93,"LINESTRING (576934.6 6817485.6, 576932 6817468..."
99,44005,03C0000002000836847,Bras de l'Esches,BDCarthage,2017/01/24 16:06:18.918,Validé,NaN,NaN,None,RGF93 / Lambert 93,"LINESTRING (646007.2 6895385.2, 646002.5 68953..."
218,82697,03C0000002000869169,Cours d'Eau 02 de la Commune de Percy,BDCarthage,2017/01/24 16:06:18.918,Validé,NaN,NaN,None,RGF93 / Lambert 93,"LINESTRING (399057.6 6876424.2, 399051.1 68764..."


In [7]:
sites = gpd.read_file("données hackaton/Sites/Sites.shp")
sites.head(3)

,Code,Code_N,Sandre_WT,Sandre_24,Libellé,Riviere,Secteur,Qualité,Méthode,But,...,Strahler,Code_Hyd_R,Code_Hyd_T,D_Source,BV_Km2,Q_Mna5,Q_Module,Altitude,Ombrage,geometry
0,1,1,3080420.0,NaN,Alfortville,Seine,Seine entrée Paris,Bonne,In-Situ,Bis,...,6,----0010,F4900010,400.79651,30634.922,64.0,201.0,34,0.0,POINT (657288.122 6855433.395)
1,10,10,3183000.0,NaN,Oissel,Seine,Seine estuaire,Bonne,In-Situ,Bis,...,7,----0010,H5010010,641.15710,71645.344,195.0,519.0,65,0.0,POINT (563788.885 6919000.796)
2,11,11,3083780.0,NaN,Chatou,Seine,Seine sortie Paris,Bonne,In-Situ,None,...,7,----0010,F7120010,453.23291,44352.273,97.0,297.0,39,0.0,POINT (638787.901 6866365.535)


## Etape 1

Graphe des stations avec les relations amont/aval entre elles (après on ajoutera dans les métadata les afluents sur lesquels elles sont). On va remonter depui l'exutoire : station initiale 